### Test with AutoGluon 

Na de classificatie van een returing customer > cluster van de klanten (customer profiling). Deze variabelen meenemen in de features. En eventueel verschillende modellen trainen op de variabelen? 

In [1]:
import pandas as pd
import numpy as np
print("numpy version:", np.__version__) # 1.6.2 needed for autogluon
from autogluon.tabular import TabularPredictor
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt


numpy version: 1.26.4


c:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
transactions = pd.read_csv(r"C:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\BigData_assignment1\data\original_data\transactions_2016_2017.csv")
train_labels = pd.read_csv(r"C:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\BigData_assignment1\data\original_data\data\customer_clv_train.csv")
test_labels =  pd.read_csv(r"C:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\BigData_assignment1\data\original_data\data\customer_clv_test.csv")     


C:\Users\Marte\AppData\Local\Temp\ipykernel_15388\1845120952.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv(r"C:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\BigData_assignment1\data\original_data\transactions_2016_2017.csv")


In [3]:
features = pd.read_csv(r"C:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\BigData_assignment1\data\features_enriched.csv")
print(features.shape) # 145 739 rijen (komt overeen met aantal unieke klanten in transactions)
print(features.info())


(145739, 161)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145739 entries, 0 to 145738
Columns: 161 entries, cust_id to expected_avg_sales
dtypes: float64(146), int64(12), object(3)
memory usage: 179.0+ MB
None


In [4]:
# Mergen met revenue 2018-2019 (om de target variable te hebben)
master_train = train_labels.merge(features, on="cust_id", how="left").fillna(0) #train labels als basis 
master_train.head()

,cust_id,revenue_2018_2019,n_items,n_sales,n_returns,first_purchase,last_purchase,n_items_non_returned,n_sales_non_returned,total_revenue_net,...,prod_print_n_unique,prod_comfort_sole_dominant_share,prod_comfort_sole_n_unique,prod_comfort_wear_dominant_share,prod_comfort_wear_n_unique,prod_clasp_dominant_share,prod_clasp_n_unique,prob_alive,pred_num_purchases,expected_avg_sales
0,klantwj2374mzmab,209.85,2,1,1,2016-01-01,2016-01-01,1.0,1.0,83.30,...,0.0,0.0,0.0,0.0,0.0,0.500000,2.0,1.000000,0.011254,0.000000
1,a63atwr2ig2jfprr,82.93,4,2,3,2016-01-01,2017-06-21,1.0,1.0,53.96,...,1.0,0.0,0.0,1.0,1.0,0.666667,2.0,0.724322,0.026478,63.250871
2,zr7ihbfbi6gcy2tz,89.95,2,2,0,2016-01-01,2017-01-27,2.0,2.0,83.70,...,0.0,1.0,1.0,1.0,1.0,0.500000,2.0,0.658666,0.024078,48.008387
3,dt7cthjqnjmkbiu6,0.00,4,2,1,2016-01-01,2017-05-24,3.0,2.0,350.92,...,0.0,0.0,0.0,0.0,0.0,0.500000,3.0,0.713269,0.026074,310.739524
4,etcrrgcbrzfovyzj,0.00,6,2,3,2016-01-01,2017-01-28,3.0,2.0,144.50,...,0.0,0.0,0.0,0.0,0.0,1.000000,1.0,0.659199,0.024097,72.016934


In [5]:
# Splits het VOLLEDIGE dataframe
train_set, val_set = train_test_split(
    master_train,                  # Hier geef je de hele tabel mee (77 kolommen)
    test_size=0.2, 
    random_state=42,
    stratify=(master_train["revenue_2018_2019"] > 0)
)

# Nu bevatten train_set en val_set allebei 77 kolommen
print(f"Train set vorm: {train_set.shape}")
print(f"Validation set vorm: {val_set.shape}")


Train set vorm: (93272, 162)
Validation set vorm: (23319, 162)


In [6]:
# Test set voorbereiden (features mergen met test_labels)
test_set_final = test_labels.merge(features, on="cust_id", how="left").fillna(0)

# Bewaar de cust_id's voor later (die heb je nodig voor je submission file)
test_cust_ids = test_set_final["cust_id"]

# Verwijder kolommen die het model niet mag zien (zoals de ID en de target kolom als die er per ongeluk in zit)
X_test_ag = test_set_final.drop(columns=["cust_id", "revenue_2018_2019"], errors="ignore")


In [7]:
train_ag = train_set.drop(columns=["cust_id"]).copy()
val_ag = val_set.drop(columns=["cust_id"]).copy()

In [10]:
# 3. INITIALISEER EN FIT IN ÉÉN KEER
# We gebruiken 'path' om de unieke map te forceren
predictor = TabularPredictor(
    label="revenue_2018_2019", 
    eval_metric="mean_absolute_error",
).fit(
    train_data=train_ag,
    tuning_data=val_ag,
    use_bag_holdout=True,  # We hebben al een aparte validation set, dus we hoeven geen extra holdout set te maken
    presets="high_quality",
    time_limit= 10600, 
    hyperparameter_tune_kwargs={
        "num_trials": 20,
        "scheduler": "local",
        "searcher": "auto",
    },
    hyperparameters={
        "CAT": {},
}
)
# 4. Toon het resultaat om te zien of het gelukt is
print(predictor.leaderboard(val_ag))

No path specified. Models will be saved in: "AutogluonModels\ag-20260511_200900"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.2
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       4.37 GB / 15.72 GB (27.8%)
Disk Space Avail:   132.12 GB / 475.28 GB (27.8%)
Presets specified: ['high_quality']
Setting dynamic_stacking from 'auto' to False. Reason: Skip dynamic_stacking when use_bag_holdout is enabled. (use_bag_holdout=True)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 10600s
AutoGluon will save models to "c:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\BigData_assignment1\cleaned_repo\AutogluonM

                       model  score_test  score_val          eval_metric  \
0    CatBoost_BAG_L1\T3_FULL  -60.803076        NaN  mean_absolute_error   
1    CatBoost_BAG_L1\T4_FULL  -61.431501        NaN  mean_absolute_error   
2   WeightedEnsemble_L3_FULL  -61.431501        NaN  mean_absolute_error   
3   WeightedEnsemble_L2_FULL  -61.431501        NaN  mean_absolute_error   
4    CatBoost_BAG_L1\T2_FULL  -61.452134        NaN  mean_absolute_error   
5    CatBoost_BAG_L1\T1_FULL  -61.506379        NaN  mean_absolute_error   
6         CatBoost_BAG_L1\T4  -61.816592 -62.581370  mean_absolute_error   
7        WeightedEnsemble_L3  -61.816592 -61.816592  mean_absolute_error   
8        WeightedEnsemble_L2  -61.816592 -61.816592  mean_absolute_error   
9         CatBoost_BAG_L1\T1  -61.873327 -62.624826  mean_absolute_error   
10        CatBoost_BAG_L1\T3  -61.885032 -62.594140  mean_absolute_error   
11        CatBoost_BAG_L1\T2  -61.993712 -62.708446  mean_absolute_error   

    pred_ti

In [11]:
importance = predictor.feature_importance(val_ag)
print(importance.head(20))

Computing feature importance via permutation shuffling for 160 features using 5000 rows with 5 shuffle sets...


	30.75s	= Expected runtime (6.15s per shuffle set)
	14.74s	= Actual runtime (Completed 5 of 5 shuffle sets)


                                   importance    stddev       p_value  n  \
n_items                              1.923027  0.399408  2.110246e-04  5   
purchase_frequency                   0.830143  0.058474  2.934730e-06  5   
recency_ratio                        0.811437  0.139050  9.954697e-05  5   
pred_num_purchases                   0.588740  0.161904  6.222247e-04  5   
customer_lifetime_days               0.556914  0.075499  3.955661e-05  5   
n_sales                              0.468897  0.087479  1.388649e-04  5   
last_purchase                        0.448949  0.074348  8.704733e-05  5   
total_revenue_net                    0.426199  0.123768  7.653087e-04  5   
days_since_last_return               0.425183  0.021257  7.471535e-07  5   
prob_alive                           0.412548  0.084782  2.025019e-04  5   
top_month_share                      0.370400  0.023992  2.100420e-06  5   
avg_days_between_purchases           0.299710  0.050583  9.377505e-05  5   
n_unique_bra

In [12]:
# Voorspel op de validatieset
y_val_true = val_set["revenue_2018_2019"]
y_val_pred = predictor.predict(val_ag.drop(columns=["revenue_2018_2019"]))

# Bereken metrieken
mae_val = mean_absolute_error(y_val_true, y_val_pred)
spearman_val, _ = spearmanr(y_val_true, y_val_pred)

autogluon_metrics = pd.DataFrame([
    {
        "model_name": "AutoGluon",
        "AUC": np.nan,
        "MAE": mae_val,
        "Spearman": spearman_val
    }
])
#autogluon_metrics.to_csv("autogluon_model_comparison.csv", index=False)

print(f"Validatie MAE: {mae_val}")
print(f"Validatie Spearman: {spearman_val}")
print("AutoGluon metrics saved to autogluon_model_comparison_enriched.csv")


Validatie MAE: 61.431501057673415
Validatie Spearman: 0.3754816862074191
AutoGluon metrics saved to autogluon_model_comparison_enriched.csv


In [13]:
# Submission maken met AutoGluon predictions op de test set

autogluon_pred_test = predictor.predict(X_test_ag)
autogluon_pred_test = np.clip(autogluon_pred_test, 0, None)

submission = pd.DataFrame({
    "cust_id": test_cust_ids,
    "predicted_revenue_2018_2019": autogluon_pred_test
})

submission_name = "submission_autogluon.csv"
submission.to_csv(submission_name, index=False)

print("Saved:", submission_name)
print("Prediction summary")
print("min :", autogluon_pred_test.min())
print("max :", autogluon_pred_test.max())
print("mean:", autogluon_pred_test.mean())

submission.head()


Saved: submission_autogluon.csv
Prediction summary
min : 0.0
max : 517.09033
mean: 26.05855


,cust_id,predicted_revenue_2018_2019
0,2dfoualegmpt6x2h,3.185787e+02
1,d2q2stjpnzld7a4r,2.774453e+01
2,cojscuqlpylhclv2,7.001619e-07
3,vntezlhi2ryvxk6m,9.823336e+01
4,jgy4ytjkdr2b75wf,1.163666e+02


## Two-stage log model

In [49]:
from datetime import datetime
from pathlib import Path

from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

def safe_spearman(y_true, y_pred):
    corr, _ = spearmanr(y_true, y_pred)
    return 0.0 if pd.isna(corr) else corr

def prepare_boosting_frames(train_df, val_df, test_df):
    train_out = train_df.copy()
    val_out = val_df.copy()
    test_out = test_df.copy()
    cat_cols = []

    for col in train_out.columns:
        if pd.api.types.is_datetime64_any_dtype(train_out[col]):
            train_out[col] = train_out[col].astype("int64") // 10**9
            val_out[col] = val_out[col].astype("int64") // 10**9
            test_out[col] = test_out[col].astype("int64") // 10**9
            continue

        if train_out[col].dtype == "object":
            parsed_train = pd.to_datetime(train_out[col], errors="coerce")
            parsed_val = pd.to_datetime(val_out[col], errors="coerce")
            parsed_test = pd.to_datetime(test_out[col], errors="coerce")
            parse_ratio = parsed_train.notna().mean()

            if parse_ratio > 0.9:
                train_out[col] = parsed_train.astype("int64") // 10**9
                val_out[col] = parsed_val.astype("int64") // 10**9
                test_out[col] = parsed_test.astype("int64") // 10**9
            else:
                train_out[col] = train_out[col].fillna("missing").astype(str).astype("category")
                val_out[col] = val_out[col].fillna("missing").astype(str).astype("category")
                test_out[col] = test_out[col].fillna("missing").astype(str).astype("category")
                cat_cols.append(col)

    return train_out, val_out, test_out, cat_cols


In [50]:
label = "revenue_2018_2019"
feature_cols_stage2 = features_to_keep.copy() if "features_to_keep" in globals() else [col for col in train_ag.columns if col != label]

train_stage2 = train_ag[feature_cols_stage2 + [label]].copy()
val_stage2 = val_ag[feature_cols_stage2 + [label]].copy()
X_test_stage2 = X_test_ag[feature_cols_stage2].copy()

train_stage2["revenue_binary"] = (train_stage2[label] > 0).astype(int)
val_stage2["revenue_binary"] = (val_stage2[label] > 0).astype(int)

buyers_train = train_stage2[train_stage2[label] > 0].copy()
buyers_val = val_stage2[val_stage2[label] > 0].copy()

train_buyers_log = buyers_train[feature_cols_stage2 + [label]].copy()
val_buyers_log = buyers_val[feature_cols_stage2 + [label]].copy()
train_buyers_log[label] = np.log1p(train_buyers_log[label])
val_buyers_log[label] = np.log1p(val_buyers_log[label])

X_train_cls = train_stage2[feature_cols_stage2].copy()
y_train_cls = train_stage2["revenue_binary"].copy()
X_val_cls = val_stage2[feature_cols_stage2].copy()
y_val_cls = val_stage2["revenue_binary"].copy()

X_val_buyers = buyers_val[feature_cols_stage2].copy()
y_val_buyers = buyers_val[label].copy()

X_train_tree, X_val_tree, X_test_tree, cat_feature_cols = prepare_boosting_frames(
    X_train_cls,
    X_val_cls,
    X_test_stage2,
)

X_train_buyers_tree, X_val_buyers_tree, X_test_buyers_tree, _ = prepare_boosting_frames(
    buyers_train[feature_cols_stage2],
    buyers_val[feature_cols_stage2],
    X_test_stage2,
)

print("Aantal features:", len(feature_cols_stage2))
print("Train rows:", len(train_stage2), "| Val rows:", len(val_stage2))
print("Positive buyers train:", len(buyers_train), "| val:", len(buyers_val))
print("Categorische features voor boosting:", cat_feature_cols)


Aantal features: 160
Train rows: 93272 | Val rows: 23319
Positive buyers train: 34126 | val: 8532
Categorische features voor boosting: []


In [24]:
cat_clf = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric="AUC",
    loss_function="Logloss",
    random_seed=42,
    verbose=False,
)
cat_clf.fit(X_train_tree, y_train_cls, cat_features=cat_feature_cols, eval_set=(X_val_tree, y_val_cls), use_best_model=True)
cat_val_proba = cat_clf.predict_proba(X_val_tree)[:, 1]

lgbm_clf = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
)
lgbm_clf.fit(X_train_tree, y_train_cls)
lgbm_val_proba = lgbm_clf.predict_proba(X_val_tree)[:, 1]

clf_train_ag = X_train_cls.copy()
clf_train_ag["revenue_binary"] = y_train_cls.values
clf_val_ag = X_val_cls.copy()
clf_val_ag["revenue_binary"] = y_val_cls.values

ag_classifier_path = f"AutogluonModels/ag-binary-{datetime.now():%Y%m%d_%H%M%S_%f}"
predictor_binary = TabularPredictor(
    label="revenue_binary",
    problem_type="binary",
    eval_metric="roc_auc",
    path=ag_classifier_path,
).fit(
    train_data=clf_train_ag,
    tuning_data=clf_val_ag,
    presets="medium_quality",
    time_limit=1800,
    num_bag_folds=0,
    num_stack_levels=0,
    dynamic_stacking=False,
    fit_weighted_ensemble=False,
)
ag_val_proba = predictor_binary.predict_proba(X_val_cls).iloc[:, 1].to_numpy()

classifier_results = pd.DataFrame([
    {"model_name": "CatBoostClassifier", "AUC": roc_auc_score(y_val_cls, cat_val_proba)},
    {"model_name": "LGBMClassifier", "AUC": roc_auc_score(y_val_cls, lgbm_val_proba)},
    {"model_name": "AutoGluonBinary", "AUC": roc_auc_score(y_val_cls, ag_val_proba)},
]).sort_values("AUC", ascending=False).reset_index(drop=True)

classifier_val_probas = {
    "CatBoostClassifier": cat_val_proba,
    "LGBMClassifier": lgbm_val_proba,
    "AutoGluonBinary": ag_val_proba,
}
classifier_test_probas = {
    "CatBoostClassifier": cat_clf.predict_proba(X_test_tree)[:, 1],
    "LGBMClassifier": lgbm_clf.predict_proba(X_test_tree)[:, 1],
    "AutoGluonBinary": predictor_binary.predict_proba(X_test_stage2).iloc[:, 1].to_numpy(),
}

best_classifier_name = classifier_results.iloc[0]["model_name"]
best_buy_proba_val = classifier_val_probas[best_classifier_name]
best_buy_proba_test = classifier_test_probas[best_classifier_name]

print(classifier_results)
print("Beste classifier:", best_classifier_name)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.2
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       1.76 GB / 15.72 GB (11.2%)
Disk Space Avail:   133.59 GB / 475.28 GB (28.1%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "c:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\BigData_assignment1\cleaned_repo\AutogluonModels\ag-binary-20260511_174720_046268"
Train Data Rows:    93272
Train Data Columns: 160
Tuning Data Rows:    23319
Tuning Data Columns: 160
Label Column:       revenue_binary
Problem Type:       binary
Preprocessing data ...
Selected class <--> l

           model_name       AUC
0  CatBoostClassifier  0.727896
1     AutoGluonBinary  0.727748
2      LGBMClassifier  0.727232
Beste classifier: CatBoostClassifier


In [25]:
cat_reg_log = CatBoostRegressor(
    iterations=700,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=False,
)
cat_reg_log.fit(X_train_buyers_tree, train_buyers_log[label], cat_features=cat_feature_cols, eval_set=(X_val_buyers_tree, val_buyers_log[label]), use_best_model=True)
cat_reg_val = np.clip(np.expm1(cat_reg_log.predict(X_val_buyers_tree)), 0, None)

ag_regressor_path = f"AutogluonModels/ag-buyers-log-cat-{datetime.now():%Y%m%d_%H%M%S_%f}"
predictor_buyers_log = TabularPredictor(
    label=label,
    eval_metric="mean_absolute_error",
    path=ag_regressor_path,
).fit(
    train_data=train_buyers_log,
    tuning_data=val_buyers_log,
    presets="medium_quality",
    time_limit=2400,
    hyperparameters={"CAT": {}},
    num_bag_folds=0,
    num_stack_levels=0,
    dynamic_stacking=False,
    fit_weighted_ensemble=False,
)
ag_reg_val = np.clip(np.expm1(predictor_buyers_log.predict(X_val_buyers)), 0, None)

regressor_results = pd.DataFrame([
    {
        "model_name": "CatBoostRegressor_log",
        "MAE_buyers": mean_absolute_error(y_val_buyers, cat_reg_val),
        "Spearman_buyers": safe_spearman(y_val_buyers, cat_reg_val),
    },
    {
        "model_name": "AutoGluon_CAT_log",
        "MAE_buyers": mean_absolute_error(y_val_buyers, ag_reg_val),
        "Spearman_buyers": safe_spearman(y_val_buyers, ag_reg_val),
    },
]).sort_values(["MAE_buyers", "Spearman_buyers"], ascending=[True, False]).reset_index(drop=True)

regressor_val_preds = {
    "CatBoostRegressor_log": cat_reg_val,
    "AutoGluon_CAT_log": ag_reg_val,
}
regressor_test_preds = {
    "CatBoostRegressor_log": np.clip(np.expm1(cat_reg_log.predict(X_test_buyers_tree)), 0, None),
    "AutoGluon_CAT_log": np.clip(np.expm1(predictor_buyers_log.predict(X_test_stage2)), 0, None),
}

best_regressor_name = regressor_results.iloc[0]["model_name"]
best_amount_pred_val_buyers = regressor_val_preds[best_regressor_name]
best_amount_pred_test = regressor_test_preds[best_regressor_name]

print(regressor_results)
print("Beste regressor:", best_regressor_name)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.2
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       2.23 GB / 15.72 GB (14.2%)
Disk Space Avail:   132.44 GB / 475.28 GB (27.9%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 2400s
AutoGluon will save models to "c:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\BigData_assignment1\cleaned_repo\AutogluonModels\ag-buyers-log-cat-20260511_175359_377084"
Train Data Rows:    34126
Train Data Columns: 160
Tuning Data Rows:    8532
Tuning Data Columns: 160
Label Column:       revenue_2018_2019
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique lab

              model_name  MAE_buyers  Spearman_buyers
0  CatBoostRegressor_log  108.176360         0.357788
1      AutoGluon_CAT_log  108.253364         0.356420
Beste regressor: CatBoostRegressor_log


In [26]:
best_amount_pred_val_full = np.clip(np.expm1(predictor_buyers_log.predict(X_val_cls)), 0, None) if best_regressor_name == "AutoGluon_CAT_log" else np.clip(np.expm1(cat_reg_log.predict(X_val_tree)), 0, None)
two_stage_pred_val = np.clip(best_buy_proba_val * best_amount_pred_val_full, 0, None)
two_stage_pred_test = np.clip(best_buy_proba_test * best_amount_pred_test, 0, None)

two_stage_results = pd.DataFrame([
    {
        "classifier": best_classifier_name,
        "regressor": best_regressor_name,
        "MAE": mean_absolute_error(val_stage2[label], two_stage_pred_val),
        "Spearman": safe_spearman(val_stage2[label], two_stage_pred_val),
    }
])

submission_two_stage = pd.DataFrame({
    "cust_id": test_set_final["cust_id"].values,
    "predicted_revenue_2018_2019": two_stage_pred_test,
})

submission_two_stage_name = "submission_two_stage_log.csv"
submission_two_stage.to_csv(submission_two_stage_name, index=False)

print(two_stage_results)
print("Saved:", submission_two_stage_name)
submission_two_stage.head()


           classifier              regressor        MAE  Spearman
0  CatBoostClassifier  CatBoostRegressor_log  70.848496  0.406974
Saved: submission_two_stage_log.csv


,cust_id,predicted_revenue_2018_2019
0,2dfoualegmpt6x2h,302.480180
1,d2q2stjpnzld7a4r,81.526432
2,cojscuqlpylhclv2,21.464562
3,vntezlhi2ryvxk6m,109.969111
4,jgy4ytjkdr2b75wf,120.690002


## Two-stage tuning and blending

In [51]:
# Volledige validatie/test-voorspellingen per regressor op alle klanten

regressor_val_preds_full = {
    "CatBoostRegressor_log": np.clip(np.expm1(cat_reg_log.predict(X_val_tree)), 0, None),
    "AutoGluon_CAT_log": np.clip(np.expm1(predictor_buyers_log.predict(X_val_cls)), 0, None),
}
regressor_test_preds_full = {
    "CatBoostRegressor_log": np.clip(np.expm1(cat_reg_log.predict(X_test_tree)), 0, None),
    "AutoGluon_CAT_log": np.clip(np.expm1(predictor_buyers_log.predict(X_test_stage2)), 0, None),
}

combo_rows = []
combo_predictions_val = {}
combo_predictions_test = {}

for clf_name, buy_proba_val in classifier_val_probas.items():
    buy_proba_test = classifier_test_probas[clf_name]
    for reg_name, amount_pred_val in regressor_val_preds_full.items():
        amount_pred_test = regressor_test_preds_full[reg_name]
        pred_val = np.clip(buy_proba_val * amount_pred_val, 0, None)
        pred_test = np.clip(buy_proba_test * amount_pred_test, 0, None)
        combo_name = f"{clf_name}__{reg_name}"

        combo_rows.append({
            "classifier": clf_name,
            "regressor": reg_name,
            "MAE": mean_absolute_error(val_stage2[label], pred_val),
            "Spearman": safe_spearman(val_stage2[label], pred_val),
        })
        combo_predictions_val[combo_name] = pred_val
        combo_predictions_test[combo_name] = pred_test

combo_results = pd.DataFrame(combo_rows).sort_values(["MAE", "Spearman"], ascending=[True, False]).reset_index(drop=True)
best_combo = combo_results.iloc[0]
best_combo_name = f"{best_combo['classifier']}__{best_combo['regressor']}"
best_combo_pred_val = combo_predictions_val[best_combo_name]
best_combo_pred_test = combo_predictions_test[best_combo_name]

print(combo_results)
print("Beste two-stage combinatie:", best_combo_name)


           classifier              regressor        MAE  Spearman
0  CatBoostClassifier  CatBoostRegressor_log  70.848496  0.406974
1      LGBMClassifier  CatBoostRegressor_log  70.880523  0.407270
2  CatBoostClassifier      AutoGluon_CAT_log  70.887754  0.407545
3     AutoGluonBinary  CatBoostRegressor_log  70.912436  0.406544
4      LGBMClassifier      AutoGluon_CAT_log  70.925312  0.407779
5     AutoGluonBinary      AutoGluon_CAT_log  70.961858  0.407053
Beste two-stage combinatie: CatBoostClassifier__CatBoostRegressor_log


In [52]:
# Zoek alpha in prediction = (p ** alpha) * amount

best_clf_name = best_combo["classifier"]
best_reg_name = best_combo["regressor"]

alpha_rows = []
alpha_predictions_val = {}
alpha_predictions_test = {}

buy_proba_val = classifier_val_probas[best_clf_name]
buy_proba_test = classifier_test_probas[best_clf_name]
amount_pred_val = regressor_val_preds_full[best_reg_name]
amount_pred_test = regressor_test_preds_full[best_reg_name]

for alpha in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 1.6, 1.8, 2.0]:
    pred_val = np.clip((buy_proba_val ** alpha) * amount_pred_val, 0, None)
    pred_test = np.clip((buy_proba_test ** alpha) * amount_pred_test, 0, None)
    alpha_rows.append({
        "alpha": alpha,
        "MAE": mean_absolute_error(val_stage2[label], pred_val),
        "Spearman": safe_spearman(val_stage2[label], pred_val),
    })
    alpha_predictions_val[alpha] = pred_val
    alpha_predictions_test[alpha] = pred_test

alpha_results = pd.DataFrame(alpha_rows).sort_values(["MAE", "Spearman"], ascending=[True, False]).reset_index(drop=True)
best_alpha = float(alpha_results.iloc[0]["alpha"])
best_alpha_pred_val = alpha_predictions_val[best_alpha]
best_alpha_pred_test = alpha_predictions_test[best_alpha]

print(alpha_results)
print("Beste alpha:", best_alpha)


    alpha        MAE  Spearman
0     2.0  63.934358  0.410503
1     1.8  64.553510  0.410182
2     1.6  65.441210  0.409754
3     1.4  66.691446  0.409160
4     1.2  68.432551  0.408290
5     1.1  69.543183  0.407701
6     1.0  70.848496  0.406974
7     0.9  72.384470  0.406047
8     0.8  74.197958  0.404876
9     0.7  76.341837  0.403351
10    0.6  78.886870  0.401331
11    0.5  81.919880  0.398553
12    0.4  85.555270  0.394705
Beste alpha: 2.0


In [53]:
# Blend beste alpha-two-stage met bestaande AutoGluon reduced voorspelling

autogluon_base_val = np.clip(np.asarray(predictor_reduced.predict(val_reduced.drop(columns=[label]))), 0, None)
autogluon_base_test = np.clip(np.asarray(predictor_reduced.predict(X_test_stage2)), 0, None)

blend_rows = []
blend_predictions_val = {}
blend_predictions_test = {}

for weight_ag in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    weight_two_stage = 1.0 - weight_ag
    pred_val = np.clip(weight_ag * autogluon_base_val + weight_two_stage * best_alpha_pred_val, 0, None)
    pred_test = np.clip(weight_ag * autogluon_base_test + weight_two_stage * best_alpha_pred_test, 0, None)

    blend_rows.append({
        "weight_autogluon": weight_ag,
        "weight_two_stage": weight_two_stage,
        "MAE": mean_absolute_error(val_stage2[label], pred_val),
        "Spearman": safe_spearman(val_stage2[label], pred_val),
    })
    blend_predictions_val[weight_ag] = pred_val
    blend_predictions_test[weight_ag] = pred_test

blend_results = pd.DataFrame(blend_rows).sort_values(["MAE", "Spearman"], ascending=[True, False]).reset_index(drop=True)
best_weight_ag = float(blend_results.iloc[0]["weight_autogluon"])
best_blend_pred_val = blend_predictions_val[best_weight_ag]
best_blend_pred_test = blend_predictions_test[best_weight_ag]

blend_summary = pd.DataFrame([
    {
        "model_name": "AutoGluon_reduced",
        "MAE": mean_absolute_error(val_stage2[label], autogluon_base_val),
        "Spearman": safe_spearman(val_stage2[label], autogluon_base_val),
    },
    {
        "model_name": f"TwoStage_best_combo_alpha_{best_alpha}",
        "MAE": mean_absolute_error(val_stage2[label], best_alpha_pred_val),
        "Spearman": safe_spearman(val_stage2[label], best_alpha_pred_val),
    },
    {
        "model_name": f"Blend_AG_{best_weight_ag:.1f}_TS_{1.0 - best_weight_ag:.1f}",
        "MAE": mean_absolute_error(val_stage2[label], best_blend_pred_val),
        "Spearman": safe_spearman(val_stage2[label], best_blend_pred_val),
    },
]).sort_values(["MAE", "Spearman"], ascending=[True, False]).reset_index(drop=True)

submission_blend = pd.DataFrame({
    "cust_id": test_set_final["cust_id"].values,
    "predicted_revenue_2018_2019": best_blend_pred_test,
})
submission_blend_name = "submission_two_stage_blend.csv"
submission_blend.to_csv(submission_blend_name, index=False)

print(blend_results)
print()
print(blend_summary)
print("Saved:", submission_blend_name)


NameError: name 'predictor_reduced' is not defined

## Optional: retrain best stage-2 regressor longer

In [ ]:
# Retrain enkel als je eerst combo_results, alpha_results en blend_summary hebt gerund.
# Deze cel traint het beste stage-2 bedragmodel langer op basis van validatiekeuze.

best_stage2_model = best_reg_name
print("Beste stage-2 regressor volgens huidige validatie:", best_stage2_model)

if best_stage2_model == "AutoGluon_CAT_log":
    ag_regressor_path_long = f"AutogluonModels/ag-buyers-log-cat-long-{datetime.now():%Y%m%d_%H%M%S_%f}"
    predictor_buyers_log_long = TabularPredictor(
        label=label,
        eval_metric="mean_absolute_error",
        path=ag_regressor_path_long,
    ).fit(
        train_data=train_buyers_log,
        tuning_data=val_buyers_log,
        presets="medium_quality",
        time_limit=5400,
        hyperparameters={"CAT": {}},
        num_bag_folds=0,
        num_stack_levels=0,
        dynamic_stacking=False,
        fit_weighted_ensemble=False,
    )
    long_reg_val = np.clip(np.expm1(predictor_buyers_log_long.predict(X_val_cls)), 0, None)
    print("Long AutoGluon buyer-regressor ready")
    print("MAE all customers after long retrain is not auto-evaluated here; plug it into the alpha/blend cells if desired.")
else:
    cat_reg_log_long = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=6,
        loss_function="MAE",
        eval_metric="MAE",
        random_seed=42,
        verbose=200,
    )
    cat_reg_log_long.fit(
        X_train_buyers_tree,
        train_buyers_log[label],
        cat_features=cat_feature_cols,
        eval_set=(X_val_buyers_tree, val_buyers_log[label]),
        use_best_model=True,
    )
    long_reg_val = np.clip(np.expm1(cat_reg_log_long.predict(X_val_tree)), 0, None)
    print("Long CatBoost buyer-regressor ready")
    print("MAE all customers after long retrain is not auto-evaluated here; plug it into the alpha/blend cells if desired.")
